# PyTO topology check
Run **one** benchmark problem with the same settings as `topopt_run_benchmarks.py`, view the topology, and compare objective/volume fraction against `Results_Reference` and the last run (`2026-08-31`).

Change `PROBLEM` / `METHOD` in the next cell and re-run all.

In [ ]:
import os, sys, csv, glob, time
os.chdir(os.path.abspath(os.path.join(os.getcwd())))
sys.path.insert(0, os.path.join(os.getcwd(), "src"))
import matplotlib; matplotlib.use("module://matplotlib_inline.backend_inline") if "ipykernel" in sys.modules else None
import matplotlib.pyplot as plt
import numpy as np
import pyvista; pyvista.OFF_SCREEN = True

from pyto.topopt.common import TO_METHODS
from pyto.examples_benchmarks.topopt_structural_benchmarks import StructuralTOExamples, getStructuralTOProblem
from pyto.examples_benchmarks.topopt_thermal_benchmarks import ThermalTOExamples, getThermalTOProblem
from pyto.physics.structural.hex_structural_fea import HexStructuralFEA
from pyto.physics.thermal.hex_thermal_fea import HexThermalFEA
import pyto.solve.numpy_backend as lin_solv
from pyto.topopt.drivers.mma import topopt_mma
from pyto.topopt.drivers.oc import topopt_optimality_criteria

# ---- choose the problem here ----
PROBLEM = ThermalTOExamples.HeatPlate       # e.g. ThermalTOExamples.BridgeThermal, StructuralTOExamples.Mitchell
METHOD  = TO_METHODS.DENSITYMMA             # DENSITYMMA or DENSITYOCM
MAX_ITERS = None                            # None -> to_params.MaxIterations (benchmark default)
REFERENCE_DIRS = {"Reference": "Results/Results_Reference", "2026-08-31": "Results/Results_2026-08-31"}

In [ ]:
def build(problem):
    if problem in ThermalTOExamples:
        mesh, mat, bc, ebf, p = getThermalTOProblem(problem)
        fe = HexThermalFEA(mesh=mesh, mat_prop=mat, bc=bc, solver=lin_solv.Solvers.PARDISO, rtol=1e-8, elem_body_force=ebf)
    else:
        mesh, mat, bc, ebf, p = getStructuralTOProblem(problem)
        fe = HexStructuralFEA(mesh=mesh, mat_prop=mat, bc=bc, solver=lin_solv.Solvers.PARDISO, rtol=1e-8, elem_body_force=ebf)
    return fe, p

fe, to_params = build(PROBLEM)
iters = MAX_ITERS or to_params.MaxIterations
print(f"{PROBLEM.name}: ndof={fe.bc.num_dofs}, objective={to_params.Objective[0].name}, constraints={[(c[0].name, c[2]) for c in to_params.Constraints]}, max iters={iters}")

In [ ]:
t0 = time.time()
if METHOD == TO_METHODS.DENSITYMMA:
    u, history, success, err, nFEAs = topopt_mma(fe, to_params=to_params, maxMMAIterations=iters, binarize_topology=True, print_progress=False)
else:
    u, history, success, err, nFEAs = topopt_optimality_criteria(fe, to_params=to_params, maxIterations=iters, binarize_topology=True, print_progress=False)
elapsed = time.time() - t0
print(f"success={success}  msg={err}  FEAs={nFEAs}  time={elapsed:.1f}s")
print(f"final objective={history['objective'][-1]:.6g}  volfrac={history['volfrac'][-1]:.4f}")

In [ ]:
def lookup(base):
    rows = []
    for f in glob.glob(f"{base}/**/{METHOD.name}_summary.csv", recursive=True):
        for r in csv.DictReader(open(f)):
            if r["name"] == PROBLEM.name:
                rows.append(r)
    return rows[0] if rows else None

print(f"{'run':12s} {'objective':>14s} {'volfrac':>9s} {'#FEAs':>6s}  success")
print(f"{'this run':12s} {history['objective'][-1]:14.6g} {history['volfrac'][-1]:9.4f} {nFEAs:6d}  {success}")
for label, base in REFERENCE_DIRS.items():
    r = lookup(base)
    if r is None: print(f"{label:12s}  (no row)"); continue
    try: print(f"{label:12s} {float(r['objective']):14.6g} {float(r['volfrac']):9.4f} {r['#FEAs']:>6s}  {r['success']}")
    except ValueError: print(f"{label:12s}  failed: {r['errorMsg'][:60]}")

In [ ]:
def topology(ax, mesh, title):
    rho = np.asarray(mesh.elemPseudoDensity).ravel()
    xyz = mesh.elem_centers
    z0 = xyz[:, 2].min()
    sl = np.isclose(xyz[:, 2], z0)
    dx = mesh.elem_size[0] if np.ndim(mesh.elem_size) else mesh.elem_size
    sc = ax.scatter(xyz[sl, 0], xyz[sl, 1], c=1 - rho[sl], cmap="gray", vmin=0, vmax=1, marker="s", s=6)
    ax.set_aspect("equal"); ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
topology(axes[0], fe.mesh, f"{PROBLEM.name} / {METHOD.name}  J={history['objective'][-1]:.4g}  vf={history['volfrac'][-1]:.3f}")
axes[1].plot(history["objective"], label="objective"); axes[1].set_xlabel("iteration"); axes[1].set_ylabel("objective")
ax2 = axes[1].twinx(); ax2.plot(history["volfrac"], "r--", label="volfrac"); ax2.set_ylabel("volfrac")
axes[1].set_title("convergence"); plt.tight_layout(); plt.show()

In [ ]:
from PIL import Image
fig, axes = plt.subplots(1, len(REFERENCE_DIRS), figsize=(5 * len(REFERENCE_DIRS), 5))
for ax, (label, base) in zip(np.atleast_1d(axes), REFERENCE_DIRS.items()):
    hits = glob.glob(f"{base}/**/{METHOD.name}/{PROBLEM.name}.png", recursive=True)
    if hits: ax.imshow(Image.open(hits[0]))
    ax.set_title(f"{label} (saved image)"); ax.axis("off")
plt.show()